# 01 - LangChain Guardrails

This notebook walks through the LangChain-layer guardrails in `src/guardrails_demo/langchain_guardrails/`: input validation, output validation, PII detection/redaction, topic guardrails, content safety, structured output validation, and tool guardrails.

No API key is required - the default chat model is an offline stub (`guardrails_demo.models.StubChatModel`) unless `GOOGLE_API_KEY` is set.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

## Input validation

Deterministic checks (empty, too short, too long, wrong type) run before any model call - see `docs/01_guardrails_fundamentals.md` for why.

In [2]:
from guardrails_demo.langchain_guardrails.input_validation import validate_input

for sample in ['What is a Python list?', '', 'x' * 3000, 12345]:
    result = validate_input(sample)
    print(repr(sample)[:40], '->', 'ALLOWED' if result.allowed else f'BLOCKED ({result.reason})')

'What is a Python list?' -> ALLOWED
'' -> BLOCKED (Input is empty or whitespace only.)
'xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx -> BLOCKED (Input exceeds the 2000 character limit.)
12345 -> BLOCKED (Expected text input, received int.)


## Output validation

In [3]:
from guardrails_demo.langchain_guardrails.output_validation import validate_output
from guardrails_demo.models import get_chat_model

model = get_chat_model()
response = model.invoke('What is a Python list?')
print('Model response:', response)
result = validate_output(response)
print('Verdict:', 'ALLOWED' if result.allowed else f'BLOCKED ({result.reason})')

Model response: [stub-response] Here is a placeholder answer for: What is a Python list?
Verdict: ALLOWED


## PII detection and redaction

Regex-based detection is educational, not a complete DLP solution (see `docs/02_langchain_guardrails.md`).

In [4]:
from guardrails_demo.langchain_guardrails.pii_guard import detect_pii, redact_pii

text = 'Contact John at john@example.com or 555-123-4567.'
result = detect_pii(text)
print('Detected:', result.metadata.get('matches'))
print('Redacted:', redact_pii(text))

Detected: {'email': ['john@example.com'], 'phone': ['555-123-4567']}
Redacted: Contact John at [EMAIL_REDACTED] or [PHONE_REDACTED].


## Topic guardrail

Keyword matching escalates to an LLM classifier only when the cheap check is inconclusive.

In [5]:
from guardrails_demo.langchain_guardrails.topic_guard import topic_guard

for text in ['What is a Python list?', "Tell me today's cricket score."]:
    result = topic_guard(text)
    print(text, '->', 'ALLOWED' if result.allowed else f'BLOCKED ({result.reason})')

What is a Python list? -> ALLOWED
Tell me today's cricket score. -> BLOCKED (Request looks unrelated to the Python programming domain.)


## Content safety

In [6]:
from guardrails_demo.langchain_guardrails.safety_guard import safety_guard, SAFE_TEST_CASES, UNSAFE_TEST_CASES

for name, text in {**SAFE_TEST_CASES, **UNSAFE_TEST_CASES}.items():
    result = safety_guard(text)
    print(f'[{name}]', 'ALLOWED' if result.allowed else f'BLOCKED ({result.reason})')

[python_question] ALLOWED
[general_question] ALLOWED
[violence_request] BLOCKED (Request matches unsafe category: violence)
[self_harm_request] BLOCKED (Request matches unsafe category: self_harm)
[illicit_request] BLOCKED (Request matches unsafe category: illicit_behavior)


## Structured output validation

In [7]:
from guardrails_demo.langchain_guardrails.structured_output_guard import validate_structured_output
from guardrails_demo.schemas import ProductReview

good = {'sentiment': 'positive', 'summary': 'Great build quality.', 'confidence': 0.92}
bad = {'sentiment': 'great', 'summary': 'Nice.', 'confidence': 5}
for sample in (good, bad):
    result = validate_structured_output(ProductReview, sample)
    print(sample, '->', 'ALLOWED' if result.allowed else f'BLOCKED ({result.reason})')

{'sentiment': 'positive', 'summary': 'Great build quality.', 'confidence': 0.92} -> ALLOWED
{'sentiment': 'great', 'summary': 'Nice.', 'confidence': 5} -> BLOCKED (('sentiment',): Input should be 'positive', 'negative' or 'neutral'; ('confidence',): Input should be less than or equal to 1)


## Tool guardrails

In [8]:
from guardrails_demo.langchain_guardrails.tool_guard import validate_tool_arguments, check_tool_allowlist
from guardrails_demo.schemas import CalculatorArguments

print(validate_tool_arguments(CalculatorArguments, {'operation': 'add', 'left': 2, 'right': 3}))
print(validate_tool_arguments(CalculatorArguments, {'operation': 'divide', 'left': 4, 'right': 0}))
print(check_tool_allowlist('shell_exec', ['calculator', 'weather_lookup']))

allowed=True reason=None category='tool_arguments_valid' confidence=1.0 modified_input=None metadata={'parsed': {'operation': 'add', 'left': 2.0, 'right': 3.0}}
allowed=False reason="('right',): Value error, cannot divide by zero" category='tool_arguments_invalid' confidence=1.0 modified_input=None metadata={}
allowed=False reason="Tool 'shell_exec' is not in the allowlist ['calculator', 'weather_lookup']." category='tool_not_allowed' confidence=1.0 modified_input=None metadata={}


## Next

See `02_langgraph_guardrails.ipynb` for the stateful, multi-step guardrails built on top of these.